# 6 — The two stages together, and the demo

Notebooks 4 and 5 measured each stage on its own. Stage 2 in particular was scored on
**gold ADE sentences** — as though Stage 1 were perfect.

It is not. This notebook chains the two and measures what that assumption costs:

```
      sentence
         |
    [ Stage 1 gate ]  --- "not an ADE" --->  no entities, done
         |
      "ADE"
         |
    [ Stage 2 tagger ]  --->  DRUG and EFFECT spans
```

The gate makes two kinds of mistake, and they do different damage:

- **a miss** — a real ADE sentence rejected. Stage 2 never sees it, so its entities are
  lost and nothing downstream can recover them.
- **a false alarm** — a non-ADE sentence passed through. Stage 2 tags it anyway, inventing
  entities that should not exist.

Everything here runs on CPU from the two saved checkpoints, in about ten seconds.

In [1]:
import sys, json, textwrap
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
pd.set_option("display.max_colwidth", 90)
print("project root:", ROOT)

project root: E:\CSE\NLP Project


---

## 6.1 Setting up the measurement

Both stages are scored over the **same 3,133-sentence Stage 1 test split**, holding all
1,612 gold entities. That only works because both stages share one split (notebook 1) — a
sentence with no annotation is a sentence whose correct answer is "no entities", which is
exactly what a gate rejection should produce.

In [2]:
from src.bio import entity_prf, strict_entities, to_bio
from src.vocab import is_indexable

stage1_test = pd.read_parquet(ROOT / "data" / "splits" / "stage1_test.parquet")
stage2_test = pd.read_parquet(ROOT / "data" / "splits" / "stage2_test.parquet")

spans_by_text = {r["text"]: json.loads(r["spans"]) for _, r in stage2_test.iterrows()}

words, gold = [], []
for text in stage1_test["text"]:
    tokens, tags = to_bio(text, [tuple(s) for s in spans_by_text.get(text, [])], strict=False)
    kept = [(t, g) for t, g in zip(tokens, tags) if is_indexable(t)]
    words.append([t for t, _ in kept])
    gold.append(strict_entities([g for _, g in kept]))

print(f"{len(stage1_test):,} test sentences")
print(f"{sum(bool(g) for g in gold):,} of them carry annotations")
print(f"{sum(len(g) for g in gold):,} gold entities in total")

3,133 test sentences
640 of them carry annotations
1,612 gold entities in total


---

## 6.2 What the gate does

In [3]:
import time
from src.pipeline import load_gate, load_tagger

t0 = time.perf_counter()
gate = load_gate()
decisions = gate.classify(stage1_test["text"].tolist())
accepted = [i for i, (_, is_ade) in enumerate(decisions) if is_ade]
print(f"gate ran over {len(stage1_test):,} sentences in {time.perf_counter() - t0:.1f}s\n")

truth = stage1_test["label"].tolist()
passed = [bool(is_ade) for _, is_ade in decisions]

confusion = pd.DataFrame(
    [[sum(1 for t, p in zip(truth, passed) if t == 1 and p),
      sum(1 for t, p in zip(truth, passed) if t == 1 and not p)],
     [sum(1 for t, p in zip(truth, passed) if t == 0 and p),
      sum(1 for t, p in zip(truth, passed) if t == 0 and not p)]],
    index=["really an ADE", "really not an ADE"],
    columns=["gate says ADE", "gate says not-ADE"])
display(confusion)

kept_ade = confusion.iloc[0, 0]
print(f"\n{len(accepted):,} sentences reach Stage 2.")
print(f"  {kept_ade} are real ADEs  -> recall {kept_ade / sum(truth):.1%}")
print(f"  {len(accepted) - kept_ade} are not -> precision {kept_ade / len(accepted):.1%}")
print(f"  {confusion.iloc[0, 1]} real ADEs were rejected and can never be recovered.")

gate ran over 3,133 sentences in 7.0s



,gate says ADE,gate says not-ADE
really an ADE,521,119
really not an ADE,130,2363



651 sentences reach Stage 2.
  521 are real ADEs  -> recall 81.4%
  130 are not -> precision 80.0%
  119 real ADEs were rejected and can never be recovered.


---

## 6.3 Oracle vs pipeline

Now the same tagger is scored twice over the same 3,133 sentences:

- **oracle** — it is handed every gold ADE sentence, the setting notebook 5 used;
- **pipeline** — the gate decides what it sees, which is the realistic setting.

In [4]:
tagger = load_tagger()

# Pipeline: only the sentences the gate accepted.
tagged = dict(zip(accepted, tagger.tag([words[i] for i in accepted])))
pipeline_pred = [strict_entities(tagged[i]) if i in tagged else [] for i in range(len(words))]

# Oracle: every sentence that really is an ADE, whatever the gate thought.
positives = [i for i in range(len(words)) if gold[i]]
otagged = dict(zip(positives, tagger.tag([words[i] for i in positives])))
oracle_pred = [strict_entities(otagged[i]) if i in otagged else [] for i in range(len(words))]

rows = []
for setting, predicted in [("oracle (Stage 1 assumed perfect)", oracle_pred),
                           ("pipeline (Stage 1 decides)", pipeline_pred)]:
    precision, recall, f1 = entity_prf(gold, predicted)
    rows.append({"setting": setting, "strict entity-F1": round(f1, 4),
                 "precision": round(precision, 3), "recall": round(recall, 3),
                 "entities predicted": sum(len(p) for p in predicted)})

comparison = pd.DataFrame(rows).set_index("setting")
display(comparison)

drop = comparison["strict entity-F1"].iloc[0] - comparison["strict entity-F1"].iloc[1]
print(f"\nerror propagation cost: -{drop:.4f} entity-F1")

,strict entity-F1,precision,recall,entities predicted
setting,,,,
oracle (Stage 1 assumed perfect),0.8308,0.854,0.809,1527
pipeline (Stage 1 decides),0.6891,0.699,0.680,1569



error propagation cost: -0.1417 entity-F1


**0.8308 → 0.6891.** A tagger that looks like it gets five entities in six right delivers
roughly two in three once a real classifier decides what it sees.

That gap *is* the finding. It is not a flaw in either model — each one performs exactly as
measured. It is what happens when two imperfect stages are chained, and it is invisible if
you only ever report per-stage scores.

Look at which side moves: precision falls from 0.854 to 0.699 and recall from 0.809 to
0.680. Both suffer, for the two different reasons named at the top — rejected ADE sentences
cost recall, and wrongly accepted ones cost precision by producing entities out of nothing.

---

## 6.4 Whose fault are the failures?

A sentence is a **failure** when the pipeline's entity set for it differs from the gold
set. Splitting those by what went wrong says where effort would go next.

In [5]:
gate_miss = gate_false_alarm = tagger_error = correct = 0

for i in range(len(words)):
    has_gold = bool(gold[i])
    was_passed = i in tagged

    if set(pipeline_pred[i]) == set(gold[i]):
        correct += 1
    elif has_gold and not was_passed:
        gate_miss += 1                 # real ADE rejected; entities unrecoverable
    elif not has_gold and was_passed:
        gate_false_alarm += 1          # not an ADE, but tagged anyway
    else:
        tagger_error += 1              # gate was right, the tagger was not

failures = gate_miss + gate_false_alarm + tagger_error
breakdown = pd.DataFrame([
    {"what went wrong": "gate miss - a real ADE was rejected", "sentences": gate_miss},
    {"what went wrong": "gate false alarm - a non-ADE was tagged", "sentences": gate_false_alarm},
    {"what went wrong": "tagger error - gate right, wrong entities", "sentences": tagger_error},
]).set_index("what went wrong")
breakdown["share of failures"] = (breakdown["sentences"] / failures).map("{:.1%}".format)
display(breakdown)

print(f"\n{failures:,} of {len(words):,} test sentences fail "
      f"({failures / len(words):.1%}); {correct:,} are exactly right.")
print(f"{gate_miss + gate_false_alarm:,} of the {failures:,} failures "
      f"({(gate_miss + gate_false_alarm) / failures:.0%}) are the GATE's, not the tagger's.")

,sentences,share of failures
what went wrong,,
gate miss - a real ADE was rejected,119,27.4%
gate false alarm - a non-ADE was tagged,130,30.0%
"tagger error - gate right, wrong entities",185,42.6%



434 of 3,133 test sentences fail (13.9%); 2,699 are exactly right.
249 of the 434 failures (57%) are the GATE's, not the tagger's.


**Most failures belong to Stage 1.** Those sentences never had a chance: either the tagger
was never shown them, or it was shown something it should not have been. That is the same
conclusion the oracle-vs-pipeline gap reached, arriving from a different direction — and it
says a better gate would buy more than a better tagger.

---

## 6.5 Where the model struggles: negation and hedging

A sentence can name a drug and a side effect and still not report an adverse event:

> *"The patient was then treated with sertraline **without** experiencing any incontinence
> episodes."*

Both entities are there. The relation between them is negated. Getting this wrong is not a
small error — it inverts the meaning.

To measure it, 615 test sentences containing a negation or hedging cue were selected **by
rule, from the sentence text alone** — no label and no prediction chose a sentence — and
frozen. That makes it a measurement rather than a collection of anecdotes.

| Group | Sentences | ADE rate |
|---|---|---|
| full test split | 3,133 | 20.4% |
| contains a negation cue (`not`, `no`, `without`, `failed to`, `ruled out`, ...) | 272 | 9.2% |
| contains a hedging cue (`may`, `suggest`, `possible`, `could`, ...) | 382 | 24.6% |
| **either — the challenge subset** | **615** | 19.0% |
| neither | 2,518 | 20.8% |

The ADE rate is similar with cues (19.0%) and without (20.8%), so a score difference
between the groups is not just a difference in class balance.

Scoring all four gates on that subset turns the embedding comparison into a second
question: *do better vectors help on the hard sentences specifically?*

In [6]:
runs = pd.read_csv(ROOT / "results" / "runs.csv").drop_duplicates("run_id", keep="last")
runs = runs.set_index("run_id")

LADDER = {"3": "E0 random", "4": "E1 GloVe", "5": "E2 Word2Vec", "6": "E3 FastText"}

rows = []
for run, name in LADDER.items():
    m = json.loads(runs.loc[f"13-{run}", "metrics_json"])
    low, high = m["ci95_cue_minus_no_cue_macro_f1"]
    rows.append({"gate": name,
                 "full split": round(m["full_macro_f1"], 4),
                 "with a cue": round(m["cue_macro_f1"], 4),
                 "no cue": round(m["no_cue_macro_f1"], 4),
                 "cue - no cue": round(m["cue_minus_no_cue_macro_f1"], 4),
                 "95% interval": f"[{low:+.3f}, {high:+.3f}]"})

pd.DataFrame(rows).set_index("gate")

,full split,with a cue,no cue,cue - no cue,95% interval
gate,,,,,
E0 random,0.7441,0.6711,0.7579,-0.0869,"[-0.140, -0.037]"
E1 GloVe,0.7942,0.7676,0.8003,-0.0328,"[-0.080, +0.010]"
E2 Word2Vec,0.8609,0.8254,0.8691,-0.0437,"[-0.086, -0.004]"
E3 FastText,0.8785,0.8584,0.8831,-0.0247,"[-0.064, +0.013]"


**Every gate does worse on sentences containing a cue.** The `cue - no cue` column is
negative for all four. The intervals are 95% bootstrap intervals for that difference, each
group resampled independently.

Two things to take from it:

**Better vectors help on the hard sentences too.** E3's score on cue sentences (0.858) is
above E0's score on *easy* ones (0.758). The whole curve lifts.

**But the gap does not close.** E3 still loses 0.025 to the cue sentences. Two of the four
intervals (E1 and E3) cross zero, so those rows are not individually conclusive. What is
convincing is that all four point the same way.

That fits what negation actually is: the cue word is often far from the drug-effect pair it
applies to, and whether it negates *that relation* or something else entirely is the whole
question. Compare the two examples the demo ships:

- *"...treated with sertraline **without** experiencing any incontinence episodes."* —
  `without` negates the relation. **Not an ADE.**
- *"A 53-year-old male, **without** any prior history of psychosis, developed
  schizophrenia 4 days after starting bromocriptine."* — `without` negates the patient's
  history, not the relation. **Still an ADE.**

Same cue word, opposite answers. Better word vectors do not resolve that; it needs
structure the model does not have.

---

## 6.6 The demo

Everything above runs on the frozen test split. The demo runs the same pipeline on **any
text you type**, which is a different problem: nothing has been tokenised, sentence-split
or decoded in advance.

`src/pipeline.py` does that, following the same rules the numbers above were measured
under — argmax for the gate decision, Stage 2 only on accepted sentences, strict entity
reading — so what the demo highlights is what the scorer would have counted.

In [7]:
from src.pipeline import EXAMPLES, MODEL_LABEL, load_pipeline

t0 = time.perf_counter()
pipeline = load_pipeline()
print(f"{MODEL_LABEL}")
print(f"both models loaded in {time.perf_counter() - t0:.1f}s on CPU\n")

for example in EXAMPLES:
    result = pipeline.analyse(example.text)[0]
    verdict = "ADE    " if result.is_ade else "not ADE"
    agrees = "OK" if result.is_ade == example.ade else "WRONG"
    print(f"[{verdict}] confidence {result.confidence:.0%}  ({agrees} - corpus says "
          f"{'ADE' if example.ade else 'not ADE'})")
    print(f"  {example.text}")
    for entity in result.entities:
        print(f"      {entity.label:7s} {entity.text}")
    print()

BiLSTM gate + BiLSTM-CRF tagger
both models loaded in 0.1s on CPU

[ADE    ] confidence 98%  (OK - corpus says ADE)
  A 55-year-old woman presented an episode of acute urticaria and labial angioedema 60 minutes after ingesting 500 mg of cloxacillin for a skin abscess.
      EFFECT  acute urticaria
      EFFECT  labial angioedema
      DRUG    cloxacillin

[ADE    ] confidence 94%  (OK - corpus says ADE)
  A case of toxic hepatitis caused by combination therapy with methotrexate and etretinate in the treatment of severe psoriasis is presented in a 47-year-old woman.
      EFFECT  toxic hepatitis
      DRUG    methotrexate
      DRUG    etretinate

[not ADE] confidence 94%  (OK - corpus says not ADE)
  The patient was then treated with sertraline without experiencing any incontinence episodes.



[ADE    ] confidence 99%  (OK - corpus says ADE)


  A 53-year-old male, without any prior history of psychosis, developed schizophrenia 4 days after starting low-dose bromocriptine therapy for a macroprolactinoma.
      EFFECT  schizophrenia
      DRUG    bromocriptine

[not ADE] confidence 97%  (OK - corpus says not ADE)
  A 23-year-old woman with metastatic Sertoli-Leydig cell tumor was treated with cisplatin, vinblastine, and bleomycin.



The same five sentences rendered the way the web app shows them — drug in blue, effect in
orange, drawn from the character offsets the tokenizer kept back in notebook 2.

In [8]:
import html as html_lib
from IPython.display import HTML

COLOURS = {"DRUG": "37, 99, 235", "EFFECT": "217, 70, 0"}

def render(result):
    parts, cursor = [], 0
    for entity in result.entities:
        parts.append(html_lib.escape(result.text[cursor:entity.start]))
        rgb = COLOURS[entity.label]
        parts.append(f'<mark style="background: rgba({rgb}, 0.18); '
                     f'border-bottom: 2px solid rgb({rgb}); color: inherit; '
                     f'padding: 0.05em 0.2em; border-radius: 0.25em;">'
                     f'{html_lib.escape(entity.text)}'
                     f'<sup style="font-size:0.6em; font-weight:700; color: rgb({rgb});">'
                     f'{entity.label}</sup></mark>')
        cursor = entity.end
    parts.append(html_lib.escape(result.text[cursor:]))
    badge = ("#b91c1c" if result.is_ade else "#6b7280")
    label = "ADE" if result.is_ade else "not ADE"
    return (f'<div style="margin: 0.9em 0; line-height: 2;">'
            f'<span style="background:{badge}; color:white; padding:0.1em 0.5em; '
            f'border-radius:0.3em; font-size:0.8em;">{label}</span> '
            f'<span style="opacity:0.6; font-size:0.85em;">'
            f'confidence {result.confidence:.0%}</span><br>{"".join(parts)}</div>')

HTML("".join(render(pipeline.analyse(e.text)[0]) for e in EXAMPLES))

Note the third and fifth: the gate rejected them, so Stage 2 never ran and there are no
highlights at all. The tagger on its own would happily have marked `sertraline` and
`incontinence` in the third one — **the gate is what stops it**, and that is the argument
for having two stages rather than one tagger.

### Running the web app

```
.venv\Scripts\python -m streamlit run app\streamlit_app.py
```

It opens at <http://localhost:8501>. Type or paste any text; each sentence is judged
separately, and the sidebar shows the model's measured scores read from
`results/runs.csv`.

`app/HOW_TO_TEST.md` is a walkthrough written for someone who has never seen the project.

---

## What this notebook showed

| Question | Answer |
|---|---|
| What does chaining the stages cost? | 0.8308 → **0.6891** entity-F1 |
| Whose fault are the failures? | **most are the gate's**, not the tagger's |
| Are negated sentences harder? | yes, for every gate — better vectors lift the curve but do not close the gap |
| Does it work on new text? | yes — `app/streamlit_app.py`, both stages on CPU in ~2s |

---

### The project in four numbers

| | |
|---|---|
| embedding ablation, Stage 1 macro-F1 | 0.744 (random) → **0.879** (our FastText) |
| Stage 2 tagger, strict entity-F1 on gold ADE sentences | **0.831** |
| CRF: structurally impossible tag sequences emitted | **5** in 640 sentences |
| end to end, both stages chained | **0.689** strict entity-F1 |

The first row is the one the project set out to produce: **word vectors trained on 159,975
PubMed abstracts beat general-purpose English vectors by 0.084 macro-F1** — more than the
gain from having pretrained vectors at all.